In [1]:
!pip install torch_geometric

In [15]:
import pandas as pd
import numpy as np
import networkx as nx
import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from math import log

# ==========================================
# 1. CHARGEMENT ET RÉINDEXATION DES DONNÉES
# ==========================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = "/content/drive/MyDrive/MLNS/"
print("Chargement des données...")
node_info = pd.read_csv(path + 'node_information.csv', header=None, index_col=0)
train_data = pd.read_csv(path + 'train.txt', sep=' ', header=None, names=['source', 'target', 'label'])
test_data = pd.read_csv(path + 'test.txt', sep=' ', header=None, names=['source', 'target'])

print("Mappage des IDs de nœuds pour PyTorch Geometric...")
mapping = {original_id: new_id for new_id, original_id in enumerate(node_info.index)}

train_data['source'] = train_data['source'].map(mapping)
train_data['target'] = train_data['target'].map(mapping)
test_data['source'] = test_data['source'].map(mapping)
test_data['target'] = test_data['target'].map(mapping)

train_data = train_data.dropna().astype(int)
test_data = test_data.dropna().astype(int)

# ==========================================
# 2. RÉDUCTION DE DIMENSION (SVD)
# ==========================================
print("Réduction des 932 features textuelles avec SVD...")
svd = TruncatedSVD(n_components=64, random_state=42)
node_features_reduced = svd.fit_transform(node_info)
node_features_tensor = torch.FloatTensor(node_features_reduced)

# ==========================================
# 3. GNN : EMBEDDINGS AVANCÉS
# ==========================================
print("Préparation du GNN Avancé (PyTorch Geometric)...")

train_edges = train_data[train_data['label'] == 1]
src = torch.LongTensor(train_edges['source'].values)
dst = torch.LongTensor(train_edges['target'].values)

edge_index = torch.stack([
    torch.cat([src, dst]),
    torch.cat([dst, src])
], dim=0)

class PyG_GraphSAGEModel_Advanced(nn.Module):
    def __init__(self, in_feats, h_feats, out_feats):
        super(PyG_GraphSAGEModel_Advanced, self).__init__()
        self.conv1 = SAGEConv(in_feats, h_feats)
        self.bn1 = nn.BatchNorm1d(h_feats)
        self.conv2 = SAGEConv(h_feats, out_feats)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x, edge_idx):
        h = self.conv1(x, edge_idx)
        h = self.bn1(h)
        h = F.relu(h)
        h = self.dropout(h)
        h = self.conv2(h, edge_idx)
        return h

class LinkPredictor_Advanced(nn.Module):
    def __init__(self, in_feats):
        super().__init__()
        self.W1 = nn.Linear(in_feats * 2, in_feats)
        self.W2 = nn.Linear(in_feats, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, h_u, h_v):
        combined = torch.cat([h_u, h_v], dim=1)
        x = F.relu(self.W1(combined))
        x = self.dropout(x)
        return torch.sigmoid(self.W2(x))

embed_model = PyG_GraphSAGEModel_Advanced(in_feats=64, h_feats=128, out_feats=64)
predictor = LinkPredictor_Advanced(64)

optimizer = torch.optim.Adam(
    list(embed_model.parameters()) + list(predictor.parameters()),
    lr=0.005,
    weight_decay=1e-4
)

print("Entraînement de GraphSAGE (2000 epochs)...")
labels_tensor = torch.FloatTensor(train_data['label'].values)
train_src_all = torch.LongTensor(train_data['source'].values)
train_dst_all = torch.LongTensor(train_data['target'].values)

embed_model.train()
predictor.train()

for epoch in range(2501):
    h = embed_model(node_features_tensor, edge_index)
    pos_score = predictor(h[train_src_all], h[train_dst_all]).squeeze()
    loss = F.binary_cross_entropy(pos_score, labels_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"  Epoch {epoch:04d} | Loss: {loss.item():.4f}")

embed_model.eval()
predictor.eval()

with torch.no_grad():
    gnn_embeddings = embed_model(node_features_tensor, edge_index).numpy()

# ==========================================
# 4. EXTRACTION DES FEATURES (Version 0.835)
# ==========================================
print("Préparation de l'extraction de features...")

def extract_ultimate_features(df, graph, embeddings, svd_features):
    features = []
    degrees = dict(graph.degree())
    pagerank = nx.pagerank(graph, alpha=0.85)

    for i, row in df.iterrows():
        u, v = int(row['source']), int(row['target'])

        common_neigh = 0
        jaccard = 0
        adamic_adar = 0
        ra_index = 0

        deg_u = degrees.get(u, 0)
        deg_v = degrees.get(v, 0)

        if graph.has_node(u) and graph.has_node(v):
            neighbors = list(nx.common_neighbors(graph, u, v))
            common_neigh = len(neighbors)
            union_size = len(set(graph[u]) | set(graph[v]))
            jaccard = common_neigh / union_size if union_size > 0 else 0

            for w in neighbors:
                d_w = degrees.get(w, 0)
                if d_w > 1:
                    adamic_adar += 1 / log(d_w)
                    ra_index += 1 / d_w

        pr_prod = pagerank.get(u, 0) * pagerank.get(v, 0)

        # Le fameux Shortest Path avec le fix NodeNotFound (mais SANS remove_edge)
        try:
            sp = nx.shortest_path_length(graph, u, v)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            sp = 10

        emb_u = embeddings[u]
        emb_v = embeddings[v]
        gnn_cos = np.dot(emb_u, emb_v) / (np.linalg.norm(emb_u) * np.linalg.norm(emb_v) + 1e-8)

        svd_u = svd_features[u]
        svd_v = svd_features[v]
        hadamard_svd = svd_u * svd_v

        base_feats = [common_neigh, jaccard, adamic_adar, ra_index, deg_u, deg_v, deg_u*deg_v, sp, pr_prod, gnn_cos]
        features.append(np.concatenate([base_feats, hadamard_svd]))

    return np.array(features)

# ==========================================
# 5. ENTRAÎNEMENT STRATIFIED K-FOLD & ENSEMBLING
# ==========================================
print("Début du K-Fold Out-Of-Fold (OOF)...")

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_train_full = train_data['label'].values
train_pairs = train_data[['source', 'target']].values

test_preds = np.zeros(len(test_data))
oof_preds = np.zeros(len(train_data))

svd_dict = {i: node_features_reduced[i] for i in range(len(node_features_reduced))}

for fold, (train_idx, val_idx) in enumerate(skf.split(train_pairs, y_train_full)):
    print(f"\n--- FOLD {fold + 1}/{n_splits} ---")

    fold_train_edges = train_data.iloc[train_idx]
    fold_val_edges = train_data.iloc[val_idx]

    G_fold = nx.Graph()
    # Le fix essentiel pour éviter le plantage
    G_fold.add_nodes_from(range(len(node_features_reduced)))

    positive_train_edges = fold_train_edges[fold_train_edges['label'] == 1][['source', 'target']].values
    G_fold.add_edges_from(positive_train_edges)

    print("  Extraction features Train...")
    X_train_fold = extract_ultimate_features(fold_train_edges, G_fold, gnn_embeddings, svd_dict)
    y_train_fold = fold_train_edges['label'].values

    print("  Extraction features Val...")
    X_val_fold = extract_ultimate_features(fold_val_edges, G_fold, gnn_embeddings, svd_dict)
    y_val_fold = fold_val_edges['label'].values

    # Paramètres XGBoost de ton record
    clf = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=8,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.7,
        eval_metric='auc',
        early_stopping_rounds=50
    )

    clf.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=100
    )

    oof_preds[val_idx] = clf.predict_proba(X_val_fold)[:, 1]

    print("  Extraction features Test...")
    X_test_fold = extract_ultimate_features(test_data, G_fold, gnn_embeddings, svd_dict)
    test_preds += clf.predict_proba(X_test_fold)[:, 1] / n_splits

# ==========================================
# 6. SOUMISSION
# ==========================================
submission = pd.DataFrame({
    'ID': range(len(test_preds)),
    'Predicted': test_preds
})
submission.to_csv('submission_record.csv', index=False)
print("\nFichier 'submission_record.csv' généré ! Tu retrouves ton meilleur code.")

Mounted at /content/drive
Chargement des données...
Mappage des IDs de nœuds pour PyTorch Geometric...
Réduction des 932 features textuelles avec SVD...
Préparation du GNN Avancé (PyTorch Geometric)...
Entraînement de GraphSAGE (2000 epochs)...
  Epoch 0000 | Loss: 0.7038
  Epoch 0100 | Loss: 0.1169
  Epoch 0200 | Loss: 0.0586
  Epoch 0300 | Loss: 0.0389
  Epoch 0400 | Loss: 0.0288
  Epoch 0500 | Loss: 0.0274
  Epoch 0600 | Loss: 0.0270
  Epoch 0700 | Loss: 0.0212
  Epoch 0800 | Loss: 0.0206
  Epoch 0900 | Loss: 0.0211
  Epoch 1000 | Loss: 0.0175
  Epoch 1100 | Loss: 0.0145
  Epoch 1200 | Loss: 0.0166
  Epoch 1300 | Loss: 0.0155
  Epoch 1400 | Loss: 0.0199
  Epoch 1500 | Loss: 0.0170
  Epoch 1600 | Loss: 0.0161
  Epoch 1700 | Loss: 0.0150
  Epoch 1800 | Loss: 0.0158
  Epoch 1900 | Loss: 0.0133
  Epoch 2000 | Loss: 0.0135
  Epoch 2100 | Loss: 0.0163
  Epoch 2200 | Loss: 0.0122
  Epoch 2300 | Loss: 0.0115
  Epoch 2400 | Loss: 0.0126
  Epoch 2500 | Loss: 0.0139
Préparation de l'extraction